### PDF Chunks RAG

In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.chains.question_answering import load_qa_chain

from dotenv import load_dotenv
import os 

In [34]:
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

In [35]:
# load models
embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
llm = ChatGoogleGenerativeAI( model="gemini-3.6-flash", max_tokens=1000)

In [36]:
# load pdf
pdf_link = "data/manual.pdf"
loader = PyPDFLoader(pdf_link, extract_images=False)
pages = loader.load_and_split()

Ignoring wrong pointing object 5 0 (offset 0)
Ignoring wrong pointing object 7 0 (offset 0)
Ignoring wrong pointing object 73 0 (offset 0)
Ignoring wrong pointing object 74 0 (offset 0)


In [37]:
# separate chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=20,
    length_function=len,
    add_start_index=True
)

chunks = text_splitter.split_documents(pages)

In [38]:
# save vector DB - Chroma
db = Chroma.from_documents(chunks, embedding=embedding_model, persist_directory="text_index")
db.persist()

In [39]:
# load vector DB - Chroma
vectordb = Chroma(persist_directory="text_index", embedding_function=embedding_model)

# load retriever (top 3 relevant documents)
retriever = vectordb.as_retriever(search_kwargs={"k":3})

# building the prompt chain for the LLM call
chain = load_qa_chain(llm, chain_type="stuff")

In [44]:
def ask(question):
    context = retriever.invoke(question)
    answer = (chain({"input_documents": context, "question": question}, return_only_outputs=True))["output_text"]
    return answer, context

In [45]:
user_question = "Segundo o documento, em que seção consigo verificar se meu documento realmente é PDF/A?"
answer, context = ask(user_question)
print("Answer: ", answer)

Answer:  Com base no documento fornecido, você pode verificar se o seu documento está no formato PDF/A na seção **"5. Verificar se o documento está no formato PDF/A"**.


In [ ]:
# top 1 relevant chunk
print(context[0])

page_content='Manual PDF/A  
 
13 
 
5. Verificar se o documento está no formato PDF/A 
Abra o PDF (no exemplo foi utilizado o Adobe Reader) e observe a 
informação: “O arquivo que você abriu está em conformidade com o 
padrão PDF/A e foi aberto como somente leitura para impedir 
modificações”. 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Elaboração pela equipe do Projeto Integrador – Junta Digital' metadata={'page_label': '13', 'source': 'data/manual.pdf', 'creator': 'Safari', 'moddate': "D:20260728165850Z00'00'", 'total_pages': 13, 'title': 'criacao-pdfs', 'author': 'Alysson Machado', 'page': 12, 'producer': 'macOS Versão 26.6 (Compilação 25G72) Quartz PDFContext', 'start_index': 0, 'creationdate': "D:20260728165850Z00'00'"}
